# Reproduce paper results with the new package
This notebook re-runs the beta-decay and dipole strength workflows **using the new `smlr` package**.

- If the original datasets are available, it will fit small emulators directly from those files.
- If data are missing, it falls back to the synthetic demo to keep CI fast.
- Set `RUN_HEAVY=true` in the environment to run on the full datasets (slower).

In [14]:
from pathlib import Path
import os
import re
import numpy as np
from smlr.data import StrengthSample, StrengthDataset
from smlr.emulator import StrengthEmulator
from smlr.metrics import normalized_l2

RUN_HEAVY = os.getenv('RUN_HEAVY', 'false').lower() in {'1', 'true', 'yes'}

# Locate data roots
beta_root = Path('../beta_decay_data_Ni_80')
dipole_root = Path('../dipoles_data_all')

print(f"Run heavy workflows: {RUN_HEAVY}")
print(f"Beta data found: {beta_root.exists()}")
print(f"Dipole data found: {dipole_root.exists()}")

Run heavy workflows: False
Beta data found: True
Dipole data found: True


In [15]:
import subprocess, sys
import itertools
import matplotlib.pyplot as plt


def load_beta_dataset(max_files=6):
    """Load a small beta-decay strength dataset into a StrengthDataset."""
    if not beta_root.exists():
        return None
    samples = []
    pattern = re.compile(r"lorm_.*_([0-9.]+)_([0-9.]+)\.out")
    files = sorted(beta_root.glob("lorm_*.out"))[:max_files]
    if not files:
        return None
    for f in files:
        m = pattern.match(f.name)
        if not m:
            continue
        beta_val, alpha_val = map(float, m.groups())
        data = np.loadtxt(f)
        if data.ndim != 2 or data.shape[1] < 2:
            continue
        energy = data[:, 0]
        strength = data[:, 1]
        samples.append(StrengthSample(params=np.array([alpha_val, beta_val]), energy=energy, strength=strength, label=f.name))
    if not samples:
        return None
    return StrengthDataset(samples)


def load_dipole_dataset(max_files=6):
    """Load a small dipole strength dataset if available (expects strength_<beta>_<alpha>.out)."""
    if not dipole_root.exists():
        return None
    candidates = list(dipole_root.glob("**/strength_*_*.out"))
    candidates = sorted(candidates)[:max_files]
    if not candidates:
        return None
    samples = []
    pattern = re.compile(r"strength_([0-9.]+)_([0-9.]+)\.out")
    for f in candidates:
        m = pattern.search(f.name)
        if not m:
            continue
        beta_val, alpha_val = map(float, m.groups())
        data = np.loadtxt(f)
        if data.ndim != 2 or data.shape[1] < 2:
            continue
        energy = data[:, 0]
        strength = data[:, 1]
        samples.append(StrengthSample(params=np.array([alpha_val, beta_val]), energy=energy, strength=strength, label=f.name))
    if not samples:
        return None
    return StrengthDataset(samples)


def fit_and_eval(ds: StrengthDataset, n_components=2, point=None):
    emu = StrengthEmulator(n_components=n_components, width_mode="global", random_state=0)
    emu.fit(ds)
    if point is None:
        point = ds.parameters().mean(axis=0)
    energy_grid = np.linspace(ds.energy_grids()[0].min(), ds.energy_grids()[0].max(), 300)
    mix, pred = emu.predict(point, energy_grid)
    # Build a crude reference by nearest neighbor
    params = ds.parameters()
    idx = np.argmin(np.linalg.norm(params - point, axis=1))
    ref = np.interp(energy_grid, ds.samples[idx].energy, ds.samples[idx].strength)
    err = normalized_l2(pred, ref, energy_grid)
    return err, pred, ref, energy_grid


if RUN_HEAVY:
    beta_ds = load_beta_dataset(max_files=12)
    dipole_ds = load_dipole_dataset(max_files=12)
else:
    beta_ds = load_beta_dataset(max_files=4)
    dipole_ds = load_dipole_dataset(max_files=4)

results = {}

if beta_ds:
    err, pred, ref, energy = fit_and_eval(beta_ds, n_components=6)
    results['beta'] = {'err': err, 'pred': pred, 'ref': ref, 'energy': energy}
    print(f"Beta-decay emulator normalized L2 vs nearest spectrum: {err:.3f}")
else:
    print("Beta-decay data not found; skipping.")

if dipole_ds:
    err, pred, ref, energy = fit_and_eval(dipole_ds, n_components=6)
    results['dipole'] = {'err': err, 'pred': pred, 'ref': ref, 'energy': energy}
    print(f"Dipole emulator normalized L2 vs nearest spectrum: {err:.3f}")
else:
    print("Dipole data not found; skipping.")

if not results:
    # Fallback to synthetic demo so CI always passes
    from smlr.demo.synthetic import main as demo_main
    out = Path('runs/demo-notebook')
    demo_main(out)
    print(f"Synthetic demo complete -> {out}")


Beta-decay emulator normalized L2 vs nearest spectrum: 0.191
Dipole emulator normalized L2 vs nearest spectrum: 0.003


In [16]:
import matplotlib.pyplot as plt
from pathlib import Path

plots_dir = Path('runs/plots-notebook')
plots_dir.mkdir(parents=True, exist_ok=True)

for name, res in results.items():
    energy = res['energy']
    pred = res['pred']
    ref = res['ref']
    err = res['err']
    fig, ax = plt.subplots(figsize=(6,4))
    ax.plot(energy, ref, label='reference', color='black')
    ax.plot(energy, pred, label='emulator', linestyle='--', color='tab:red')
    ax.set_xlabel('Energy (MeV)')
    ax.set_ylabel('Strength')
    ax.set_title(f"{name.capitalize()} | normalized L2 = {err:.3f}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    outfile = plots_dir / f"{name}_comparison.png"
    fig.savefig(outfile, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved plot: {outfile}")


Saved plot: runs/plots-notebook/beta_comparison.png
Saved plot: runs/plots-notebook/dipole_comparison.png
